# Trace Acquisition — Masked AEAD (ChaCha20-Poly1305 + Trivium/TRNG)

This notebook implements the complete trace-acquisition pipeline for the
**first-order masked** ChaCha20-Poly1305 AEAD core on the CW305 (Artix-7)
FPGA target.

## Workflow Overview

| Step | Description |
|---:|---|
| 1 | Open ChipWhisperer scope and program FPGA bitstream |
| 2 | Configure clocks (PLL, ADC sync) and gain |
| 3 | Patch ITF register driver onto target object |
| 4 | Functional verification (HW vs. PyCryptodome reference) |
| 5 | Single-trace capture and visual inspection |
| 6 | Bulk fixed-vs-random TVLA trace acquisition (R-F-F-R interleaving) |
| 7 | Quick inline TVLA (Welch's $t$-test on captured data) |

## Hardware Assumptions

- **Board**: CW305 with Artix-7 100T.
- **Capture**: ChipWhisperer-Lite (or Pro / Husky).
- **Bitstream**: Masked AEAD core with ITF registers.
- **DIP switches**: J16 = 0, K16 = 1 (PLL1 → FPGA → CW ADC).
- **ADC sync**: `extclk_x4` (phase-locked to FPGA output clock).

> **Important**: Adjust `PLATFORM`, `TARGET_PLATFORM`, and bitstream path
> below to match your hardware setup.

---

## 1 — Scope Setup & FPGA Programming

The CW305 target is programmed automatically when instantiated if a `.bit`
path is provided.  Use `force=True` to reprogram after a bitstream update.

In [ ]:
import sys, os, time
import chipwhisperer as cw

# Resolve repo root robustly (works regardless of kernel cwd)
_nb_dir = os.path.dirname(os.path.abspath("__file__"))
for _c in [os.path.join(_nb_dir, ".."), _nb_dir]:
    if os.path.isdir(os.path.join(os.path.abspath(_c), "src")):
        REPO_ROOT = os.path.abspath(_c); break
else:
    REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.config import BITSTREAMS, DEFINES_FILE, DATA_COMBINED

# ---------------------------------------------------------------------------
# Platform selection
# ---------------------------------------------------------------------------
PLATFORM        = "CWLITE"     # or "CWPRO" / "CWHUSKY"
TARGET_PLATFORM = "CW305_100t" # or "CW305_35t" / "CW312T_A35"
ILA_DEBUG       = False

BITSTREAM = BITSTREAMS["aead_masked"]

# ---------------------------------------------------------------------------
# Scope initialisation (with cleanup on reconnect)
# ---------------------------------------------------------------------------
try:
    scope = cw.scope()
except Exception:
    for obj_name in ("scope", "target"):
        if obj_name in globals():
            try: globals()[obj_name].dis()
            except Exception: pass
    time.sleep(0.5)
    scope = cw.scope()
    print("[init] Scope reopened after cleanup.")

scope.default_setup(verbose=False)
scope.adc.offset = 0
scope.adc.basic_mode = "high"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2  = "disabled"

# Platform-specific FPGA identifier
if TARGET_PLATFORM == "CW312T_A35":
    scope.io.hs2 = "clkgen"
    fpga_id  = "cw312t_a35"
    platform = "ss2"
else:
    platform = "cw305"
    fpga_id  = "100t" if TARGET_PLATFORM == "CW305_100t" else "35t"

# --- Program FPGA ---
print(f"Programming FPGA: {BITSTREAM}")
target = cw.target(None, cw.targets.CW305,
                   bsfile=BITSTREAM, force=(not ILA_DEBUG),
                   platform=platform,
                   defines_files=[DEFINES_FILE])

## 2 — ADC & Clock Configuration

Critical for side-channel measurements: the ADC clock must be
**phase-locked** to the FPGA core clock to avoid sampling jitter.
On CW-Lite/Pro this is achieved by setting `adc_src = "extclk_x4"`.

In [ ]:
# --- Platform-specific ADC & PLL settings ---
if PLATFORM == "CWLITE":
    scope.adc.samples = 10000
    target.pll.pll_outfreq_set(15E6, 1)
    target._clksleeptime = 150
    scope.gain.db = 26
elif PLATFORM == "CWPRO":
    scope.adc.stream_mode = True
    scope.adc.samples = 1_200_000
    target.pll.pll_outfreq_set(10E6, 1)
    target._clksleeptime = 150
    scope.gain.db = 20
elif PLATFORM == "CWHUSKY":
    scope.adc.stream_mode = True
    scope.adc.samples = 1_200_000
    target.pll.pll_outfreq_set(15E6, 1)
    target._clksleeptime = 100
    scope.gain.db = 20

# --- Clock synchronisation ---
if TARGET_PLATFORM == "CW312T_A35":
    scope.clock.clkgen_freq = 7.37e6
    scope.io.hs2 = "clkgen"
    if PLATFORM == "CWHUSKY":
        scope.clock.clkgen_src = "system"
        scope.clock.adc_mul = 1
        scope.clock.reset_dcms()
    else:
        scope.clock.adc_src = "clkgen_x1"
    time.sleep(0.1)
    target._ss2_test_echo()
else:
    if PLATFORM == "CWHUSKY":
        scope.clock.clkgen_freq = 15e6
        scope.clock.clkgen_src = "extclk"
        scope.clock.adc_mul = 1
    else:
        # CW-Lite / Pro: lock ADC to external FPGA clock × 4
        print("CLOCK SYNC: adc_src = extclk_x4")
        scope.clock.adc_src = "extclk_x4"
        scope.clock.freq_ctr_src = "extclk"
        scope.adc.decimate = 1

scope.adc.offset = 3 if PLATFORM == "CWHUSKY" else 0

# FPGA PLL outputs
if "CW305" in TARGET_PLATFORM:
    target.vccint_set(1.0)
    target.pll.pll_enable_set(True)
    target.pll.pll_outenable_set(False, 0)
    target.pll.pll_outenable_set(True,  1)  # output 1 = core clock
    target.pll.pll_outenable_set(False, 2)

# --- Reset & verify lock ---
scope.clock.reset_adc()
time.sleep(0.5)
assert scope.clock.adc_locked, "ADC failed to lock — check FPGA clock!"

project = cw.create_project("projects/AEAD_HW_CW305.cwp", overwrite=True)
time.sleep(1)

print(f"ADC source : {scope.clock.adc_src}")
print(f"ADC locked : {scope.clock.adc_locked}")

## 3 — Register Driver & AEAD Helpers

The ITF register interface is monkey-patched onto the `target` object,
then the AEAD packing and test-vector functions are imported.

In [ ]:
from src.itf_driver import patch_target
from src.aead_helpers import (
    aead_pack_and_load, aead_load_key, aead_load_data,
    preload_constant_data, run_aead_test, run_aead_key_only,
    reference_chacha20_poly1305, get_last_trace_codes,
)

patch_target(target)
target.itf_clk_div_set(2)
print("ITF driver patched and clock divider set.")

## 4 — Functional Verification

Compare hardware output against the PyCryptodome reference for both
the default test vector and 1 000 random keys.  A single mismatch
aborts the test and indicates a hardware or configuration issue.

In [ ]:
# --- Single-shot sanity check ---
ct_ref, tag_ref = reference_chacha20_poly1305()
ct_hw, tag_hw, cycles = run_aead_test(target)
assert ct_hw == ct_ref and tag_hw == tag_ref, "Mismatch on default test vector!"
print(f"Default vector OK ({cycles} cycles).")

In [ ]:
import random
from tqdm.auto import tqdm

N_VERIFY = 1000
errors   = 0

# Pre-load constant data (nonce, AAD, PT) once
preload_constant_data(target)

for i in tqdm(range(N_VERIFY), desc="Functional verification"):
    key = random.getrandbits(256)
    ct_ref, tag_ref = reference_chacha20_poly1305(key)

    # Key-only load + execute + readback
    target.itf_set_control("RESET")
    target.itf_set_control("NULL")
    aead_load_key(target, key)
    target.it_set_next_block("NO_OP")
    target.itf_start_test()
    while not target.itf_done():
        pass
    ct_hw  = target.itf_read_ciphertext()
    tag_hw = target.itf_read_tag()

    if ct_hw != ct_ref or tag_hw != tag_ref:
        errors += 1
        print(f"\nMISMATCH at iteration {i}, key={key:064x}")
        break

if errors == 0:
    print(f"\nAll {N_VERIFY} random-key tests passed.")
else:
    print(f"\nVerification FAILED — {errors} error(s).")

## 5 — Single-Trace Capture & Inspection

Acquire a small number of traces and visualise them interactively
to verify signal quality and trigger alignment before bulk capture.

In [ ]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()

scope.arm()
run_aead_test(target, verbose=False)
ret = scope.capture()
assert not ret, "Capture timeout!"

trace = get_last_trace_codes(scope, PLATFORM)

p = figure(width=900, height=300,
           title="Single Trace (ADC codes)",
           x_axis_label="Sample", y_axis_label="ADC Code",
           tools="pan,wheel_zoom,box_zoom,reset,save")
p.line(range(len(trace)), trace.tolist(), line_color="navy", line_width=1)
show(p)
print(f"Trace length: {len(trace)} samples")

## 6 — Bulk TVLA Acquisition (R-F-F-R Interleaving)

Traces are captured in a balanced **Random–Fixed–Fixed–Random** order to
mitigate temporal drift bias.  The data is streamed in batches directly to
an HDF5 file on disk with periodic `flush()` for crash safety.

### Output format

```
/traces_fixed/traces    → (N, S) int16, LZF-compressed
/traces_random/traces   → (N, S) int16, LZF-compressed
```

> **Note**: If the capture is interrupted (Ctrl-C), the file is resized to
> whatever was captured so that downstream analysis works correctly.

In [ ]:
import random, h5py, time
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Acquisition parameters
# ---------------------------------------------------------------------------
N_TRACES_TARGET = 1_000_000
WINDOW_SIZE     = scope.adc.samples
BATCH_SIZE      = 500
MAX_RETRIES     = 5

OUTPUT_FILE = os.path.join(
    DATA_COMBINED,
    "traces_combined_TVLA_v6_1M_smpl_7_5_Mhz.h5"
)

# Fixed key (RFC 8439 test vector)
KEY_FIXED = 0x808182838485868788898A8B8C8D8E8F909192939495969798999A9B9C9D9E9F

# Pre-generate random keys
print("Pre-generating random keys...")
random_keys = [random.getrandbits(256) for _ in range(N_TRACES_TARGET)]

def capture_one(key_int):
    """Arm → execute → capture → return trace (int16) or None."""
    for attempt in range(MAX_RETRIES + 1):
        scope.arm()
        run_aead_key_only(target, key_int)
        if not scope.capture():
            tr = scope.get_last_trace()
            return (np.asarray(tr) * 65536).astype(np.int16)
        if attempt == MAX_RETRIES:
            print(f"  Capture failed after {MAX_RETRIES} retries.")
            return None
    return None

# Ensure even number
N_TRACES_TARGET -= N_TRACES_TARGET % 2
iterations = N_TRACES_TARGET // 2

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

print(f"Output : {OUTPUT_FILE}")
print(f"Target : {N_TRACES_TARGET:,} traces ({iterations:,} R-F-F-R blocks)")

preload_constant_data(target)

try:
    with h5py.File(OUTPUT_FILE, "w") as f:
        dset_f = f.create_group("traces_fixed").create_dataset(
            "traces", (N_TRACES_TARGET, WINDOW_SIZE), dtype="int16",
            compression="lzf", chunks=(BATCH_SIZE, WINDOW_SIZE),
            maxshape=(None, WINDOW_SIZE))
        dset_r = f.create_group("traces_random").create_dataset(
            "traces", (N_TRACES_TARGET, WINDOW_SIZE), dtype="int16",
            compression="lzf", chunks=(BATCH_SIZE, WINDOW_SIZE),
            maxshape=(None, WINDOW_SIZE))

        buf_f = np.zeros((BATCH_SIZE, WINDOW_SIZE), dtype=np.int16)
        buf_r = np.zeros((BATCH_SIZE, WINDOW_SIZE), dtype=np.int16)
        buf_idx, global_idx, total_captured = 0, 0, 0

        try:
            for i in tqdm(range(iterations), desc="Acquiring (R-F-F-R)"):
                tr_r1 = capture_one(random_keys[i * 2])
                tr_f1 = capture_one(KEY_FIXED)
                tr_f2 = capture_one(KEY_FIXED)
                tr_r2 = capture_one(random_keys[i * 2 + 1])

                if any(t is None for t in (tr_r1, tr_f1, tr_f2, tr_r2)):
                    continue  # discard incomplete block

                buf_r[buf_idx] = tr_r1;  buf_f[buf_idx] = tr_f1;  buf_idx += 1
                buf_f[buf_idx] = tr_f2;  buf_r[buf_idx] = tr_r2;  buf_idx += 1

                if buf_idx >= BATCH_SIZE:
                    s, e = global_idx, global_idx + buf_idx
                    dset_f[s:e] = buf_f[:buf_idx]
                    dset_r[s:e] = buf_r[:buf_idx]
                    global_idx += buf_idx
                    total_captured = global_idx
                    buf_idx = 0
                    f.flush()

        except KeyboardInterrupt:
            print("\nInterrupted — saving captured data...")
        finally:
            if buf_idx > 0:
                s, e = global_idx, global_idx + buf_idx
                dset_f[s:e] = buf_f[:buf_idx]
                dset_r[s:e] = buf_r[:buf_idx]
                total_captured += buf_idx
            dset_f.resize((total_captured, WINDOW_SIZE))
            dset_r.resize((total_captured, WINDOW_SIZE))
            print(f"Saved {total_captured:,} traces to {OUTPUT_FILE}")

except Exception as e:
    print(f"Fatal error: {e}")

## 7 — Cleanup

Disconnect scope and target to release USB handles.

In [ ]:
target.dis()
scope.dis()
print("Scope and target disconnected.")